# Qwen 2B LoRA Fine-Tuning for Text-to-SVG Generation

This notebook fine-tunes `Qwen3.5-2B-Instruct` with LoRA on a **50K CSV dataset** of (prompt, svg) pairs,
then generates 1000 SVGs from test prompts (max 8000 chars each) for Kaggle submission.

**Key settings:**
- Dataset: your 50K CSV with `prompt` and `svg` columns (no other datasets used)
- Max sequence length: 8192 tokens (to accommodate SVGs up to ~8000 chars)
- LoRA rank: 32, alpha: 64
- Output SVGs capped at 8000 characters

## 1. Installation

In [ ]:
%%capture
!pip install --upgrade pip
!pip install "unsloth[cu124-ampere-torch250] @ git+https://github.com/unslothai/unsloth.git"
!pip install --no-deps trl==0.22.2 xformers bitsandbytes accelerate peft
!pip install datasets pandas lxml

## 2. Configuration

In [1]:
import os
import re
import time
import random
import xml.etree.ElementTree as ET

import numpy as np
import pandas as pd
import torch
from datasets import Dataset

# ── Reproducibility ──
SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

print(f"Torch: {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    print(f"GPU Memory: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")

Torch: 2.10.0+cu128
CUDA available: True
GPU: NVIDIA GeForce RTX 3090
GPU Memory: 25.8 GB


In [2]:
# ── All configurable paths and hyperparameters ──
CONFIG = {
    # Paths — UPDATE THESE to match your setup
    "train_csv": "./csv/train_clean.csv",           # Your 50K CSV with 'prompt' and 'svg' columns
    "test_prompts_csv": "./csv/test.csv",   # Test prompts CSV with 'id' and 'prompt' columns
    "output_dir": "./qwen2b_svg_lora_v3",          # Where to save LoRA adapter checkpoints
    "submission_path": "./submission.csv",       # Output submission file

    # Model
    "model_name": "unsloth/Qwen3.5-2B",
    "max_seq_length": 2478,       # Must be large enough for prompt + SVG (up to ~8000 chars)
    "load_in_4bit": True,

    # LoRA
    "lora_r": 32,
    "lora_alpha": 64,
    "lora_dropout": 0,

    # Training
    "num_train_epochs": 1,
    "per_device_train_batch_size": 2,      # Keep low for long sequences
    "gradient_accumulation_steps": 8,     # Effective batch size = 16
    "learning_rate": 2e-4,
    "warmup_ratio": 0.05,
    "weight_decay": 0.01,
    "logging_steps": 10,
    "save_steps": 300,
    "eval_size": 0.01,                     # 1% holdout for eval (~500 samples)

    # Inference
    "max_new_tokens": 4096,                # Token budget for generation
    "max_svg_chars": 8000,                 # Hard cap on SVG string length
    "temperature": 0.6,
    "top_p": 0.9,
    "repetition_penalty": 1.05,
}

print("Config:")
for k, v in CONFIG.items():
    print(f"  {k}: {v}")

Config:
  train_csv: ./csv/train_clean.csv
  test_prompts_csv: ./csv/test.csv
  output_dir: ./qwen2b_svg_lora_v3
  submission_path: ./submission.csv
  model_name: unsloth/Qwen3.5-2B
  max_seq_length: 2478
  load_in_4bit: True
  lora_r: 32
  lora_alpha: 64
  lora_dropout: 0
  num_train_epochs: 1
  per_device_train_batch_size: 2
  gradient_accumulation_steps: 8
  learning_rate: 0.0002
  warmup_ratio: 0.05
  weight_decay: 0.01
  logging_steps: 10
  save_steps: 300
  eval_size: 0.01
  max_new_tokens: 4096
  max_svg_chars: 8000
  temperature: 0.6
  top_p: 0.9
  repetition_penalty: 1.05


## 3. Load and Prepare Your 50K Dataset

In [3]:
# ── Load your CSV ──
df = pd.read_csv(CONFIG["train_csv"])
print(f"Raw CSV rows: {len(df)}")
print(f"Columns: {list(df.columns)}")
print(f"\nFirst row preview:")
print(f"  prompt: {str(df['prompt'].iloc[0])[:120]}...")
print(f"  svg length: {len(str(df['svg'].iloc[0]))} chars")

Raw CSV rows: 33000
Columns: ['prompt', 'svg']

First row preview:
  prompt: The image displays a black icon with a photo-like rectangle containing a wavy line on the left side and a plus sign on t...
  svg length: 2333 chars


In [4]:
# ── Clean and filter ──
df = df.dropna(subset=["prompt", "svg"])
df["prompt"] = df["prompt"].astype(str).str.strip()
df["svg"] = df["svg"].astype(str).str.strip()

# Keep only rows where svg looks valid
df = df[df["svg"].str.lower().str.startswith("<svg")]
df = df[df["prompt"].str.len() > 0]

# Filter out SVGs that are too long (won't fit in context or exceed submission limit)
df["svg_len"] = df["svg"].str.len()
df = df[df["svg_len"] <= CONFIG["max_svg_chars"]]

print(f"After cleaning: {len(df)} rows")
print(f"SVG length stats:")
print(df["svg_len"].describe())

After cleaning: 33000 rows
SVG length stats:
count    33000.000000
mean      2218.354667
std       1651.384960
min         91.000000
25%        929.000000
50%       1799.000000
75%       3097.250000
max       7999.000000
Name: svg_len, dtype: float64


In [5]:
# ── Format into chat template for SFT ──
SYSTEM_PROMPT = (
    "You are an SVG generation assistant. Given a text description, generate valid SVG code. "
    "Output ONLY the SVG markup starting with <svg> and ending with </svg>. "
    "Use a 256x256 canvas with viewBox='0 0 256 256'. Keep the SVG compact and under 8000 characters."
)


def format_sft_text(row):
    return (
        f"<|im_start|>system\n{SYSTEM_PROMPT}<|im_end|>\n"
        f"<|im_start|>user\n{row['prompt']}<|im_end|>\n"
        f"<|im_start|>assistant\n{row['svg']}<|im_end|>"
    )


df["text"] = df.apply(format_sft_text, axis=1)

# Check token-length estimate (rough: 1 token ≈ 3.5 chars for code)
df["est_tokens"] = df["text"].str.len() / 3.5
print(f"Estimated token length stats:")
print(df["est_tokens"].describe())

# Warn about samples that might be truncated
over_limit = (df["est_tokens"] > CONFIG["max_seq_length"]).sum()
print(f"\nSamples likely exceeding {CONFIG['max_seq_length']} tokens: {over_limit}")

Estimated token length stats:
count    33000.000000
mean       764.880312
std        474.170096
min        147.428571
25%        395.357143
50%        645.714286
75%       1018.000000
max       2478.000000
Name: est_tokens, dtype: float64

Samples likely exceeding 2478 tokens: 0


In [6]:
# ── Create HuggingFace Dataset and split ──
dataset = Dataset.from_pandas(df[["text"]].reset_index(drop=True))
dataset = dataset.shuffle(seed=SEED)

splits = dataset.train_test_split(test_size=CONFIG["eval_size"], seed=SEED)
train_dataset = splits["train"]
eval_dataset = splits["test"]

print(f"Train samples: {len(train_dataset)}")
print(f"Eval samples:  {len(eval_dataset)}")
print(f"\nSample text (first 300 chars):")
print(train_dataset[0]["text"][:300])

Train samples: 32670
Eval samples:  330

Sample text (first 300 chars):
<|im_start|>system
You are an SVG generation assistant. Given a text description, generate valid SVG code. Output ONLY the SVG markup starting with <svg> and ending with </svg>. Use a 256x256 canvas with viewBox='0 0 256 256'. Keep the SVG compact and under 8000 characters.<|im_end|>
<|im_start|>use


## 4. Load Model + LoRA

In [7]:
from unsloth import FastLanguageModel

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name=CONFIG["model_name"],
    max_seq_length=CONFIG["max_seq_length"],
    dtype=None,                          # Auto-detect (float16 / bfloat16)
    load_in_4bit=CONFIG["load_in_4bit"],
)

print(f"Model loaded: {CONFIG['model_name']}")

🦥 Unsloth: Will patch your computer to enable 2x faster free finetuning.
Unsloth: Your Flash Attention 2 installation seems to be broken. Using Xformers instead. No performance changes will be seen.
🦥 Unsloth Zoo will now patch everything to make training faster!
==((====))==  Unsloth 2026.3.17: Fast Qwen3_5 patching. Transformers: 5.3.0.
   \\   /|    NVIDIA GeForce RTX 3090. Num GPUs = 1. Max memory: 24.0 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.10.0+cu128. CUDA: 8.6. CUDA Toolkit: 12.8. Triton: 3.6.0
\        /    Bfloat16 = TRUE. FA [Xformers = None. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!


The fast path is not available because one of the required library is not installed. Falling back to torch implementation. To install follow https://github.com/fla-org/flash-linear-attention#installation and https://github.com/Dao-AILab/causal-conv1d


Loading weights:   0%|          | 0/617 [00:00<?, ?it/s]

Model loaded: unsloth/Qwen3.5-2B


In [8]:
model = FastLanguageModel.get_peft_model(
    model,
    r=CONFIG["lora_r"],
    lora_alpha=CONFIG["lora_alpha"],
    lora_dropout=CONFIG["lora_dropout"],
    bias="none",
    target_modules=[
        "q_proj", "k_proj", "v_proj", "o_proj",
        "gate_proj", "up_proj", "down_proj",
    ],
    use_gradient_checkpointing="unsloth",
    random_state=SEED,
)

# Print trainable parameter count
trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
total = sum(p.numel() for p in model.parameters())
print(f"Trainable parameters: {trainable:,} / {total:,} ({100*trainable/total:.2f}%)")

Unsloth: Making `model.base_model.model.model.language_model` require gradients
Trainable parameters: 21,823,488 / 1,397,711,680 (1.56%)


## 5. Training

In [9]:
# Run this to see if you really need 8192
print(df["svg_len"].quantile([0.5, 0.75, 0.9, 0.95, 0.99]))
print(f"\nEstimated token lengths (chars / 3.5):")
print((df["svg_len"] / 3.5).quantile([0.5, 0.75, 0.9, 0.95, 0.99]))

0.50    1799.00
0.75    3097.25
0.90    4627.00
0.95    5664.05
0.99    7253.02
Name: svg_len, dtype: float64

Estimated token lengths (chars / 3.5):
0.50     514.000000
0.75     884.928571
0.90    1322.000000
0.95    1618.300000
0.99    2072.291429
Name: svg_len, dtype: float64


In [10]:
from transformers import TrainingArguments
from trl import SFTTrainer

training_args = TrainingArguments(
    output_dir=CONFIG["output_dir"],
    num_train_epochs=CONFIG["num_train_epochs"],
    per_device_train_batch_size=CONFIG["per_device_train_batch_size"],
    gradient_accumulation_steps=CONFIG["gradient_accumulation_steps"],
    learning_rate=CONFIG["learning_rate"],
    warmup_ratio=CONFIG["warmup_ratio"],
    weight_decay=CONFIG["weight_decay"],
    fp16=not torch.cuda.is_bf16_supported(),
    bf16=torch.cuda.is_bf16_supported(),
    logging_steps=CONFIG["logging_steps"],
    eval_strategy="steps",
    eval_steps=500,
    save_steps=CONFIG["save_steps"],
    save_total_limit=2,
    report_to="none",
    optim="paged_adamw_8bit",
    lr_scheduler_type="cosine",
    seed=SEED,
    dataloader_num_workers=2,
)

trainer = SFTTrainer(
    model=model,
    tokenizer=tokenizer,
    train_dataset=train_dataset,
    eval_dataset=eval_dataset,
    dataset_text_field="text",
    max_seq_length=CONFIG["max_seq_length"],
    packing=False,                            # Pack short samples together for efficiency
    args=training_args,
)

print("Trainer ready. Starting training...")
print(f"Estimated steps: ~{len(train_dataset) // (CONFIG['per_device_train_batch_size'] * CONFIG['gradient_accumulation_steps'])} per epoch")

warmup_ratio is deprecated and will be removed in v5.2. Use `warmup_steps` instead.
warmup_ratio is deprecated and will be removed in v5.2. Use `warmup_steps` instead.


Unsloth: Tokenizing ["text"] (num_proc=13):   0%|          | 0/32670 [00:00<?, ? examples/s]

Unsloth: Tokenizing ["text"] (num_proc=12):   0%|          | 0/330 [00:00<?, ? examples/s]

Trainer ready. Starting training...
Estimated steps: ~2041 per epoch


In [11]:
# ── Memory stats before training ──
if torch.cuda.is_available():
    gpu_stats = torch.cuda.get_device_properties(0)
    start_gpu_memory = round(torch.cuda.max_memory_reserved() / 1024 / 1024 / 1024, 3)
    max_memory = round(gpu_stats.total_memory / 1024 / 1024 / 1024, 3)
    print(f"GPU = {gpu_stats.name}. Max memory = {max_memory} GB.")
    print(f"{start_gpu_memory} GB of memory reserved before training.")

GPU = NVIDIA GeForce RTX 3090. Max memory = 24.0 GB.
3.967 GB of memory reserved before training.


In [12]:
train_result = trainer.train()

# ── Memory stats after training ──
if torch.cuda.is_available():
    used_memory = round(torch.cuda.max_memory_reserved() / 1024 / 1024 / 1024, 3)
    print(f"\nPeak reserved memory = {used_memory} GB.")
    print(f"Training runtime: {train_result.metrics['train_runtime']:.0f}s "
          f"({train_result.metrics['train_runtime']/60:.1f} min)")

The tokenizer has new PAD/BOS/EOS tokens that differ from the model config and generation config. The model config and generation config were aligned accordingly, being updated with the tokenizer's values. Updated tokens: {'eos_token_id': 248046}.
==((====))==  Unsloth - 2x faster free finetuning | Num GPUs used = 1
   \\   /|    Num examples = 32,670 | Num Epochs = 1 | Total steps = 2,042
O^O/ \_/ \    Batch size per device = 2 | Gradient accumulation steps = 8
\        /    Data Parallel GPUs = 1 | Total batch size (2 x 8 x 1) = 16
 "-____-"     Trainable parameters = 21,823,488 of 2,235,065,152 (0.98% trained)


Step,Training Loss,Validation Loss
500,0.351805,0.333577
1000,0.334651,0.316314
1500,0.325981,0.303824
2000,0.303895,0.299432



Peak reserved memory = 20.203 GB.
Training runtime: 39031s (650.5 min)


In [13]:
# ── Save LoRA adapter ──
os.makedirs(CONFIG["output_dir"], exist_ok=True)
trainer.save_model(CONFIG["output_dir"])
tokenizer.save_pretrained(CONFIG["output_dir"])
print(f"LoRA adapter saved to: {CONFIG['output_dir']}")

LoRA adapter saved to: ./qwen2b_svg_lora_v3


## 6. Inference Utilities

In [15]:
# ── SVG extraction and validation ──
SVG_REGEX = re.compile(r"<svg[\s\S]*?</svg>", flags=re.IGNORECASE)

ALLOWED_TAGS = {
    "svg", "g", "path", "rect", "circle", "ellipse", "line", "polyline", "polygon",
    "defs", "use", "symbol", "clipPath", "mask", "linearGradient", "radialGradient",
    "stop", "text", "tspan", "title", "desc", "style", "pattern", "marker", "filter",
    # Also allow lowercase/namespace variants
    "clippath", "lineargradient", "radialgradient",
}


def extract_svg(text):
    """Extract the first <svg>...</svg> block from generated text."""
    m = SVG_REGEX.search(text)
    if not m:
        return ""
    svg = m.group(0).strip()
    # Truncate if over the character limit
    if len(svg) > CONFIG["max_svg_chars"]:
        return ""
    return svg


def is_valid_svg(svg_text):
    """Check if SVG parses and uses only allowed tags."""
    if not svg_text:
        return False
    try:
        root = ET.fromstring(svg_text)
    except ET.ParseError:
        return False

    # Check root is svg
    local_name = root.tag.split("}")[-1] if "}" in root.tag else root.tag
    if local_name.lower() != "svg":
        return False

    # Check all elements are in allowed set
    for elem in root.iter():
        tag = elem.tag.split("}")[-1] if "}" in elem.tag else elem.tag
        if tag.lower() not in ALLOWED_TAGS:
            return False

    return True


def ensure_svg_header(svg_text):
    """Ensure the SVG has proper xmlns and dimensions."""
    if 'xmlns=' not in svg_text:
        svg_text = svg_text.replace('<svg', '<svg xmlns="http://www.w3.org/2000/svg"', 1)
    if 'viewBox' not in svg_text:
        svg_text = svg_text.replace('<svg', '<svg viewBox="0 0 256 256"', 1)
    return svg_text


def fallback_svg(prompt):
    """Generate a simple fallback SVG so the submission row isn't empty."""
    # Hash prompt to get a deterministic color
    h = hash(prompt) % 360
    return (
        '<svg xmlns="http://www.w3.org/2000/svg" width="256" height="256" viewBox="0 0 256 256">'
        '<rect x="0" y="0" width="256" height="256" fill="white"/>'
        f'<circle cx="128" cy="128" r="80" fill="hsl({h}, 60%, 50%)"/>'
        '</svg>'
    )


print("Inference utilities loaded.")

Inference utilities loaded.


In [16]:
# ── Generation function ──
FastLanguageModel.for_inference(model)


def generate_svg(prompt, max_new_tokens=None):
    """Generate an SVG from a text prompt."""
    if max_new_tokens is None:
        max_new_tokens = CONFIG["max_new_tokens"]

    chat_text = (
        f"<|im_start|>system\n{SYSTEM_PROMPT}<|im_end|>\n"
        f"<|im_start|>user\n{prompt}<|im_end|>\n"
        f"<|im_start|>assistant\n"
    )
    inputs = tokenizer(chat_text, return_tensors="pt").to(model.device)

    with torch.no_grad():
        output_ids = model.generate(
            **inputs,
            max_new_tokens=max_new_tokens,
            do_sample=True,
            temperature=CONFIG["temperature"],
            top_p=CONFIG["top_p"],
            repetition_penalty=CONFIG["repetition_penalty"],
        )

    decoded = tokenizer.decode(output_ids[0], skip_special_tokens=True)
    svg = extract_svg(decoded)

    if svg:
        svg = ensure_svg_header(svg)

    if not is_valid_svg(svg):
        svg = fallback_svg(prompt)

    return svg

In [17]:
# ── Quick sanity check ──
test_prompts_quick = [
    "a red circle",
    "a green tree with brown trunk",
    "a simple blue bird icon",
]

for p in test_prompts_quick:
    svg = generate_svg(p)
    print(f"Prompt: {p}")
    print(f"  Valid: {is_valid_svg(svg)}, Length: {len(svg)} chars")
    print(f"  Preview: {svg[:150]}...")
    print()

ValueError: Incorrect image source. Must be a valid URL starting with `http://` or `https://`, a valid path to an image file, or a base64 encoded string. Got <|im_start|>system
You are an SVG generation assistant. Given a text description, generate valid SVG code. Output ONLY the SVG markup starting with <svg> and ending with </svg>. Use a 256x256 canvas with viewBox='0 0 256 256'. Keep the SVG compact and under 8000 characters.<|im_end|>
<|im_start|>user
a red circle<|im_end|>
<|im_start|>assistant
. Failed with cannot identify image file <_io.BytesIO object at 0x714b0ecdce50>

## 7. Generate Submission (1000 SVGs)

In [ ]:
# ── Load test prompts ──
test_df = pd.read_csv(CONFIG["test_prompts_csv"])
print(f"Test prompts: {len(test_df)}")
print(f"Columns: {list(test_df.columns)}")
test_df.head()

In [ ]:
# ── Generate all SVGs ──
rows = []
invalid_count = 0
t0 = time.time()

for idx, row in test_df.iterrows():
    prompt = row["prompt"]
    svg = generate_svg(prompt)

    if not is_valid_svg(svg):
        invalid_count += 1
        svg = fallback_svg(prompt)

    # Final length safety check
    if len(svg) > CONFIG["max_svg_chars"]:
        svg = fallback_svg(prompt)
        invalid_count += 1

    rows.append({"id": row["id"], "svg": svg})

    # Progress logging
    if (idx + 1) % 100 == 0:
        elapsed = time.time() - t0
        rate = (idx + 1) / elapsed
        remaining = (len(test_df) - idx - 1) / rate
        print(f"  [{idx+1}/{len(test_df)}] {rate:.1f} samples/s, "
              f"~{remaining/60:.1f} min remaining, {invalid_count} fallbacks so far")

elapsed_min = (time.time() - t0) / 60
print(f"\nDone! Generated {len(rows)} SVGs in {elapsed_min:.1f} minutes.")
print(f"Invalid/fallback count: {invalid_count} ({100*invalid_count/len(rows):.1f}%)")

In [ ]:
# ── Save submission CSV ──
sub_df = pd.DataFrame(rows)

# Verify all SVGs are valid and within limits
sub_df["svg_len"] = sub_df["svg"].str.len()
print(f"Submission SVG length stats:")
print(sub_df["svg_len"].describe())
print(f"Max SVG length: {sub_df['svg_len'].max()} (limit: {CONFIG['max_svg_chars']})")

assert sub_df["svg_len"].max() <= CONFIG["max_svg_chars"], "Some SVGs exceed the character limit!"
assert len(sub_df) == len(test_df), "Row count mismatch!"

sub_df[["id", "svg"]].to_csv(CONFIG["submission_path"], index=False)
print(f"\nSubmission saved to: {CONFIG['submission_path']}")
print(f"Rows: {len(sub_df)}")
sub_df[["id", "svg_len"]].head(10)

## 8. (Optional) Save Merged Model for Kaggle Submission

If you need to run inference on Kaggle (no internet), save the merged model and upload it as a Kaggle dataset.

In [14]:
SAVE_MERGED = True  # Set to True when ready

if SAVE_MERGED:
    merged_dir = "./qwen2b_svg_merged"
    model.save_pretrained_merged(merged_dir, tokenizer, save_method="merged_16bit")
    print(f"Merged model saved to: {merged_dir}")
    print("Upload this folder as a Kaggle dataset for offline inference.")

Found HuggingFace hub cache directory: /home/rohan/.cache/huggingface/hub


Fetching 1 files:   0%|          | 0/1 [00:00<?, ?it/s]

Checking cache directory for required files...



Unsloth: Copying 1 files from cache to `./qwen2b_svg_merged`:   0%|                               | 0/1 [00:00<?, ?it/s]
Unsloth: Copying 1 files from cache to `./qwen2b_svg_merged`: 100%|███████████████████████| 1/1 [00:17<00:00, 17.86s/it]


Successfully copied all 1 files from cache to `./qwen2b_svg_merged`
Checking cache directory for required files...
Cache check failed: tokenizer.model not found in local cache.
Not all required files found in cache. Will proceed with downloading.



Unsloth: Preparing safetensor model files: 100%|███████████████████████████████████████| 1/1 [00:00<00:00, 14027.77it/s]

Unsloth: Merging weights into 16bit: 100%|████████████████████████████████████████████████| 1/1 [00:10<00:00, 10.14s/it]


Unsloth: Merge process complete. Saved to `/home/rohan/nyu_svg_contest/qwen2b_svg_merged`
Merged model saved to: ./qwen2b_svg_merged
Upload this folder as a Kaggle dataset for offline inference.


In [ ]:
# ── Push LoRA adapter to HuggingFace (optional) ──
PUSH_TO_HF = False  # Set to True when ready

if PUSH_TO_HF:
    HF_USERNAME = "YOUR_USERNAME"  # Change this
    HF_TOKEN = "YOUR_HF_TOKEN"    # Change this
    model.push_to_hub(f"{HF_USERNAME}/qwen2b-svg-lora", token=HF_TOKEN)
    tokenizer.push_to_hub(f"{HF_USERNAME}/qwen2b-svg-lora", token=HF_TOKEN)
    print("Pushed to HuggingFace Hub.")